## Summary 

Purpose of this notebook is to derive trait features for use in AI classifier and daily wildlfe challenges.

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
import re
import numpy as np

In [2]:
birds = pd.read_csv("BirdFuncDat.txt", sep="\t", encoding="latin1", low_memory=False)
mammals = pd.read_csv("MamFuncDat.txt",  sep="\t", encoding="latin1", low_memory=False)

In [5]:
df = pd.read_csv("wildlife_occurrences_with_location_end_status_v2_w_lat_long.csv")
df = df.query("year == 2022 ").reset_index(drop = True)
# remove extra column
df = df.drop("CommonName", axis = 1)

In [16]:
agg_cols = ["year","CommonName_clean","ScientificName","EPBCThreatStatus"]

df_agg = ((df.groupby(agg_cols)
              .agg({"occurrenceCount": "sum"}))
            .sort_values("occurrenceCount", ascending = False)
            .reset_index()
            .rename(columns = {"CommonName_clean": "CommonName"}))

In [17]:
df_agg.head()

,year,CommonName,ScientificName,EPBCThreatStatus,occurrenceCount
0,2022,Rabbit,Oryctolagus cuniculus,Present,5754
1,2022,Red Fox,Vulpes vulpes,Present,5485
2,2022,Koala,Phascolarctos cinereus,Present,4897
3,2022,Eastern Grey Kangaroo,Macropus giganteus,Present,3734
4,2022,Swamp Wallaby,Wallabia bicolor,Present,3199


In [21]:
def _norm_name(s):
    return re.sub(r"\s+", " ", str(s).strip())

# --- 1) Pick the columns we care about (exist in your screenshot) -------------
bird_cols = [
    "Scientific",
    # diet shares
    "Diet-Inv","Diet-Vend","Diet-Vect","Diet-Vfish","Diet-Vunk",
    "Diet-Scav","Diet-Fruit","Diet-Nect","Diet-Seed","Diet-PlantO","Diet-5Cat",
    # foraging strata
    "ForStrat-wataroundsurf","ForStrat-watbelowsurf","ForStrat-ground",
    "ForStrat-understory","ForStrat-midhigh","ForStrat-canopy","ForStrat-aerial",
    # activity + body mass
    "Nocturnal","BodyMass-Value"
]
mam_cols = [
    "Scientific","MSWFamilyLatin",
    "Diet-Inv","Diet-Vend","Diet-Vect","Diet-Vfish","Diet-Vunk",
    "Diet-Scav","Diet-Fruit","Diet-Nect","Diet-Seed","Diet-PlantO",
    "ForStrat-Value",              # mammals often have a composite; may be NaN
    "Activity-Nocturnal","Activity-Diurnal","Activity-Crepuscular",
    "BodyMass-Value"
]

birds_use   = birds[[c for c in bird_cols if c in birds.columns]].copy()
mammals_use = mammals[[c for c in mam_cols if c in mammals.columns]].copy()

birds_use["taxon_class_ET"]   = "Aves"
mammals_use["taxon_class_ET"] = "Mammalia"

# --- 2) Normalise names -------------------------------------------------------
birds_use["Scientific_norm"]   = birds_use["Scientific"].map(_norm_name)
mammals_use["Scientific_norm"] = mammals_use["Scientific"].map(_norm_name)

# --- 3) Combine to one table --------------------------------------------------
et = pd.concat([birds_use, mammals_use], ignore_index=True)

# --- 4) Helper: find “top” column among a set of shares ----------------------
def _top_col(row, cols):
    best_c, best_v = None, float("-inf")
    for c in cols:
        if c in row and pd.notna(row[c]):
            try:
                v = float(row[c])
                if v > best_v:
                    best_c, best_v = c, v
            except Exception:
                pass
    return best_c, (None if best_v == float("-inf") else best_v)

# label maps for readability
DIET_LABELS = {
    "Diet-Inv":"invertebrates","Diet-Vend":"vert_warm","Diet-Vect":"vert_cold",
    "Diet-Vfish":"fish","Diet-Vunk":"vertebrate_unknown","Diet-Scav":"scavenge",
    "Diet-Fruit":"fruit","Diet-Nect":"nectar","Diet-Seed":"seed","Diet-PlantO":"other_plant",
    "Diet-5Cat":"diet_5cat"
}
FORSTRAT_LABELS = {
    "ForStrat-wataroundsurf":"water_surface","ForStrat-watbelowsurf":"underwater",
    "ForStrat-ground":"ground","ForStrat-understory":"understory",
    "ForStrat-midhigh":"mid_high","ForStrat-canopy":"canopy","ForStrat-aerial":"aerial",
    "ForStrat-Value":"foraging_value"
}

diet_cols   = [c for c in et.columns if c.startswith("Diet-")]
for_cols    = [c for c in et.columns if c.startswith("ForStrat-")]

# --- 5) Derived “top” summaries ----------------------------------------------
et["diet_top_col"], et["diet_top_val"] = zip(*et.apply(lambda r: _top_col(r, diet_cols), axis=1))
et["diet_top"] = et["diet_top_col"].map(lambda c: DIET_LABELS.get(c, c) if c else None)

et["foraging_top_col"], et["foraging_top_val"] = zip(*et.apply(lambda r: _top_col(r, for_cols), axis=1))
et["foraging_top"] = et["foraging_top_col"].map(lambda c: FORSTRAT_LABELS.get(c, c) if c else None)

# --- 6) Activity summary (birds: Nocturnal 0/1; mammals: three Activity-* cols)
def _activity_label(row):
    acols = [c for c in ["Activity-Nocturnal","Activity-Diurnal","Activity-Crepuscular"] if c in row.index]
    if acols:
        c, v = _top_col(row, acols)
        if c: return c.replace("Activity-","").lower()
    if "Nocturnal" in row and pd.notna(row["Nocturnal"]):
        return "nocturnal" if float(row["Nocturnal"]) > 0 else "non-nocturnal"
    return None

et["activity_top"] = et.apply(_activity_label, axis=1)

# --- 7) Size bucket from BodyMass-Value (grams)
def _size_bucket(g):
    try:
        g = float(g)
    except Exception:
        return None
    if g < 100:    return "Small"     # mouse-sized, small birds
    if g < 1000:   return "Medium"    # blackbird/possum sized
    if g < 10000:  return "Large"     # cat/raptor sized
    return "Very Large"               # swans, kangaroos, sea-eagles, etc.

et["size_bucket"] = et["BodyMass-Value"].map(_size_bucket)

# --- 8) Normalize shares to proportions (0..1) -------------------------------
def _norm_share(x):
    try:
        v = float(x)
        return v/100.0 if v > 1.0 else v
    except Exception:
        return np.nan

for c in diet_cols + for_cols + ["Nocturnal","Activity-Nocturnal","Activity-Diurnal","Activity-Crepuscular"]:
    if c in et.columns:
        et[c+"_p"] = et[c].map(_norm_share)

# --- 9) Binary game features (safe) ------------------------------------------
# can_fly: all birds True; mammals only if Chiroptera-family hint (approx)
et["can_fly"] = (et["taxon_class_ET"] == "Aves")
if "MSWFamilyLatin" in et.columns:
    et["can_fly"] = et["can_fly"] | et["MSWFamilyLatin"].fillna("").str.contains(
        "Pteropodidae|Vespertilionidae|Molossidae|Miniopteridae|Chiroptera", case=False
    )

et["aerial_forager"] = et.get("ForStrat-aerial_p", pd.Series([np.nan]*len(et))).fillna(0).astype(float).gt(0.30)
et["ground_forager"] = et.get("ForStrat-ground_p", pd.Series([np.nan]*len(et))).fillna(0).astype(float).gt(0.30)
water_p = et[[c for c in et.columns if c.endswith(("wataroundsurf_p","watbelowsurf_p"))]].fillna(0).sum(axis=1)
et["wetland_user"] = water_p.astype(float).gt(0.30)

# nocturnal flag
noct_p = et.get("Activity-Nocturnal_p")
if noct_p is None:
    noct_p = et.get("Nocturnal_p")
et["nocturnal"] = pd.Series(noct_p).fillna(0).astype(float).gt(0.50).values

# can_swim: wetland user OR underwater > 0.30
underwater_p = et.get("ForStrat-watbelowsurf_p", pd.Series([0]*len(et))).fillna(0).astype(float)
et["can_swim"] = et["wetland_user"] | underwater_p.gt(0.30)

# manual override example (Platypus)
swim_overrides_true = {"Ornithorhynchus anatinus"}
et["can_swim"] = et.apply(
    lambda r: True if _norm_name(r.get("Scientific")) in swim_overrides_true else r["can_swim"],
    axis=1
)

# --- 10) NEW DERIVED FEATURES -------------------------------------------------
# Thresholds
DIET_T = 0.20
FOR_T  = 0.20

# Diet diversity & flags
diet_p_cols = [c for c in et.columns if c.startswith("Diet-") and c.endswith("_p")]
et["diet_diversity"] = et[diet_p_cols].gt(DIET_T).sum(axis=1)

et["eats_insects"] = et.get("Diet-Inv_p", pd.Series([0]*len(et))).gt(DIET_T)
et["eats_fruit"]   = et.get("Diet-Fruit_p", pd.Series([0]*len(et))).gt(DIET_T)
et["eats_fish"]    = et.get("Diet-Vfish_p", pd.Series([0]*len(et))).gt(DIET_T)

# Omnivore flag: any plant cat > T AND any animal cat > T
plant_cols  = ["Diet-Fruit_p","Diet-Nect_p","Diet-Seed_p","Diet-PlantO_p"]
animal_cols = ["Diet-Inv_p","Diet-Vend_p","Diet-Vect_p","Diet-Vfish_p","Diet-Vunk_p","Diet-Scav_p"]
plant_any   = et[ [c for c in plant_cols  if c in et.columns] ].fillna(0).gt(DIET_T).any(axis=1)
animal_any  = et[ [c for c in animal_cols if c in et.columns] ].fillna(0).gt(DIET_T).any(axis=1)
et["omnivore_flag"] = plant_any & animal_any

# Foraging diversity & flags
for_p_cols = [c for c in et.columns if c.startswith("ForStrat-") and c.endswith("_p")]
et["foraging_diversity"] = et[for_p_cols].gt(FOR_T).sum(axis=1)

et["forages_canopy"]      = et.get("ForStrat-canopy_p",      pd.Series([0]*len(et))).gt(FOR_T)
et["forages_understory"]  = et.get("ForStrat-understory_p",  pd.Series([0]*len(et))).gt(FOR_T)
et["forages_ground"]      = et.get("ForStrat-ground_p",      pd.Series([0]*len(et))).gt(FOR_T)
et["forages_aerial"]      = et.get("ForStrat-aerial_p",      pd.Series([0]*len(et))).gt(FOR_T)
et["forages_underwater"]  = et.get("ForStrat-watbelowsurf_p",pd.Series([0]*len(et))).gt(FOR_T)
et["forages_water_surf"]  = et.get("ForStrat-wataroundsurf_p",pd.Series([0]*len(et))).gt(FOR_T)

# Activity diversity & flags
act_cols_p = [c for c in ["Activity-Nocturnal_p","Activity-Diurnal_p","Activity-Crepuscular_p"] if c in et.columns]
if act_cols_p:
    et["activity_diversity"] = et[act_cols_p].fillna(0).gt(0.30).sum(axis=1)
    et["is_crepuscular"] = et.get("Activity-Crepuscular_p", pd.Series([0]*len(et))).gt(0.30)
    day_p  = et.get("Activity-Diurnal_p",   pd.Series([0]*len(et)))
    nite_p = et.get("Activity-Nocturnal_p", pd.Series([0]*len(et)))
    et["is_cathemeral"] = day_p.gt(0.30) & nite_p.gt(0.30)
else:
    et["activity_diversity"] = 0
    et["is_crepuscular"] = False
    et["is_cathemeral"] = False

# Body mass extras
et["bodymass_log10"] = np.where(et["BodyMass-Value"].notna() & (et["BodyMass-Value"]>0),
                                np.log10(et["BodyMass-Value"].astype(float)),
                                np.nan)
et["megafauna_20kg"] = et["BodyMass-Value"].fillna(0).astype(float).ge(20000)  # >= 20kg

# Specialist / generalist heuristics
# Normalize top values to proportion if needed
def _to_prop(x):
    try:
        v = float(x)
        return v/100.0 if v > 1.0 else v
    except Exception:
        return np.nan

et["diet_top_val_p"]     = et["diet_top_val"].map(_to_prop)
et["foraging_top_val_p"] = et["foraging_top_val"].map(_to_prop)

et["specialist_flag"] = (et["diet_top_val_p"].fillna(0).ge(0.80)) & (et["foraging_top_val_p"].fillna(0).ge(0.80))
et["generalist_flag"] = (et["diet_diversity"].ge(4)) | (et["foraging_diversity"].ge(3))

# Taxonomic hints
# Marsupial families (not exhaustive, but good coverage for AU): 
_marsupial_pat = r"(Macropodidae|Potoroidae|Phalangeridae|Petauridae|Burramyidae|Peramelidae|Dasyuridae|Vombatidae|Phascolarctidae)"
et["is_marsupial"] = et.get("MSWFamilyLatin", pd.Series([""]*len(et))).fillna("").str.contains(_marsupial_pat, case=False, regex=True)
_bat_pat = r"(Pteropodidae|Vespertilionidae|Molossidae|Miniopteridae)"
et["is_bat"] = et.get("MSWFamilyLatin", pd.Series([""]*len(et))).fillna("").str.contains(_bat_pat, case=False, regex=True)

# --- 11) Prepare a slim traits table to join to your df5 ---------------------
keep_originals = [
    "Scientific","Scientific_norm","taxon_class_ET","BodyMass-Value",
    "diet_top","foraging_top","activity_top","size_bucket",
    "can_fly","can_swim","nocturnal","aerial_forager","ground_forager","wetland_user",
    # NEW:
    "diet_diversity","omnivore_flag","eats_insects","eats_fruit","eats_fish",
    "foraging_diversity","forages_canopy","forages_understory","forages_ground","forages_aerial",
    "forages_underwater","forages_water_surf",
    "activity_diversity","is_crepuscular","is_cathemeral",
    "bodymass_log10","megafauna_20kg","specialist_flag","generalist_flag",
    "is_marsupial","is_bat"
]
traits_slim = (
    et[ [c for c in keep_originals if c in et.columns] ]
    .drop_duplicates("Scientific_norm")
    .rename(columns={"BodyMass-Value":"BodyMass_g_ET","Scientific":"Scientific_ET"})
)

# --- 12) Merge with your species list ----------------------------------------
df_use = df.copy()
df_use["Scientific_norm"] = df_use["ScientificName"].map(_norm_name)

enriched = df_use.merge(traits_slim, on="Scientific_norm", how="left")

# --- 13) Order columns for game use ------------------------------------------
front = [
    "year","CommonName","ScientificName","EPBCThreatStatus","occurrenceCount",
    "taxon_class_ET","BodyMass_g_ET","size_bucket","bodymass_log10","megafauna_20kg",
    "diet_top","foraging_top","activity_top",
    "diet_diversity","foraging_diversity","activity_diversity",
    "omnivore_flag","eats_insects","eats_fruit","eats_fish",
    "can_fly","can_swim","nocturnal","is_crepuscular","is_cathemeral",
    "aerial_forager","forages_aerial","ground_forager","forages_ground",
    "wetland_user","forages_underwater","forages_water_surf","forages_canopy","forages_understory",
    "specialist_flag","generalist_flag","is_marsupial","is_bat"
]
enriched = enriched[[c for c in front if c in enriched.columns] + [c for c in enriched.columns if c not in front]]

print(f"Rows in df: {len(df):,}")
print(f"Matched diet info for: {enriched['diet_top'].notna().sum()} species")


Rows in df: 6,014
Matched diet info for: 3988 species


/var/folders/8t/xfwg22n50z94wlpff0x0bj680000gn/T/ipykernel_25763/1207336782.py:210: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  et["is_marsupial"] = et.get("MSWFamilyLatin", pd.Series([""]*len(et))).fillna("").str.contains(_marsupial_pat, case=False, regex=True)
/var/folders/8t/xfwg22n50z94wlpff0x0bj680000gn/T/ipykernel_25763/1207336782.py:212: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  et["is_bat"] = et.get("MSWFamilyLatin", pd.Series([""]*len(et))).fillna("").str.contains(_bat_pat, case=False, regex=True)


In [24]:
enriched.head()

,year,ScientificName,EPBCThreatStatus,occurrenceCount,taxon_class_ET,BodyMass_g_ET,size_bucket,bodymass_log10,megafauna_20kg,diet_top,...,Kingdom,griisStatus,capadStatus,IBRA_REG_N,STATE,lat,lon,CommonName_clean,Scientific_norm,Scientific_ET
0,2022,Anous stolidus,Present,2,Aves,177.77,Medium,2.249858,False,fish,...,Animalia,Native,PA,NaN,NaN,NaN,NaN,Common Noddy,Anous stolidus,Anous stolidus
1,2022,Fregata ariel,Present,2,Aves,804.32,Medium,2.905429,False,invertebrates,...,Animalia,Native,PA,NaN,NaN,NaN,NaN,Lesser Frigatebird,Fregata ariel,Fregata ariel
2,2022,Fregata ariel,Present,1,Aves,804.32,Medium,2.905429,False,invertebrates,...,Animalia,Native,not protected,NaN,NaN,NaN,NaN,Lesser Frigatebird,Fregata ariel,Fregata ariel
3,2022,Onychoprion fuscatus,Present,1,NaN,NaN,NaN,NaN,NaN,NaN,...,Animalia,Native,PA,NaN,NaN,NaN,NaN,Sooty Tern,Onychoprion fuscatus,NaN
4,2022,Phaethon lepturus,Present,1,Aves,328.04,Medium,2.515927,False,fish,...,Animalia,Native,PA,NaN,NaN,NaN,NaN,White-tailed Tropicbird,Phaethon lepturus,Phaethon lepturus
